# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

---

# My Solution: **Tech Tutor** — a technical Q&A prototype

This builds on my [week 1 exercise](../../week1/W1_assessment_DJ/week1%20EXERCISE%20Solution.ipynb) and upgrades it with everything from week 2.

**Everything here runs on the OpenAI API only** — one `OpenAI` client, no Anthropic / Google / Ollama.

| Requirement | How it is done |
| --- | --- |
| Gradio UI | A `gr.ChatInterface` prototype, then a full `gr.Blocks` app |
| Streaming | Token-by-token `yield` from the chat completions stream |
| System prompt for expertise | Prompt is assembled from a **domain** + an **audience level** picker |
| Switch between models | Dropdown over four OpenAI models, all through the same client |
| **Bonus: tools** | Two tools — a docs cheat-sheet lookup, and a snippet runner that *actually executes* the code before explaining it |

The interesting part is that streaming and tool calling are combined: tool-call fragments are re-assembled from the stream, the tools run, and the answer keeps streaming afterwards.

## 1. Setup

In [1]:
# imports

import os
import io
import json
from contextlib import redirect_stdout

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization - load the key from .env and connect to OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please add OPENAI_API_KEY to your .env file")

openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [3]:
# Model switching.
# Every one of these is served by the OpenAI API, so the single client above handles all of them -
# switching model is just a different string in the API call.

MODELS = {
    "GPT-4o-mini (cheap all-rounder)": "gpt-4o-mini",
    "GPT-4.1-mini (balanced)": "gpt-4.1-mini",
    "GPT-4.1-nano (fastest)": "gpt-4.1-nano",
    "GPT-4o (most capable)": "gpt-4o",
}

DEFAULT_MODEL = "GPT-4o-mini (cheap all-rounder)"

list(MODELS)

['GPT-4o-mini (cheap all-rounder)',
 'GPT-4.1-mini (balanced)',
 'GPT-4.1-nano (fastest)',
 'GPT-4o (most capable)']

## 2. The system prompt — where the expertise comes from

Rather than one fixed system prompt, it is assembled from three pieces:

1. a **base persona** that sets the house style and explains the available tools,
2. a **domain** so the assistant answers as a specialist rather than a generalist,
3. an **audience level** so the same question gets a different answer for a beginner and for an expert.

The last two are dropdowns in the UI, so you can re-ask the same question at a different level and watch the answer change.

In [4]:
BASE_SYSTEM_PROMPT = """You are Tech Tutor, an expert technical assistant used during an LLM engineering course.

How you answer:
- Explain step by step: the idea first, then the detail.
- Use plain language. Introduce jargon only after you have defined it.
- Include a short, runnable code example whenever it makes the answer clearer.
- If the question is ambiguous, state the assumption you are making and answer anyway.
- If you do not know, say so instead of guessing.
- Always reply in markdown.

You have tools available. Use them when they make your answer more accurate:
- lookup_reference(topic): a curated cheat sheet plus a link to the official documentation.
  Use it when the user asks about a core language or library concept.
- run_python_snippet(code): actually executes a short, self-contained Python snippet and returns
  what it printed. Prefer to run a snippet before explaining what it does, then quote the real output.

Never invent tool output, and never claim you ran code that you did not run."""


EXPERTISE = {
    "General technology": "Your speciality is general software engineering and computer science.",
    "Python": "Your speciality is Python: the language itself, the standard library, idioms, packaging and the common gotchas.",
    "Data & SQL": "Your speciality is databases and SQL: schema design, query tuning, indexes, transactions and pandas for analysis.",
    "Data Science & ML": "Your speciality is data science and machine learning: numpy, pandas, scikit-learn, PyTorch, model evaluation and the maths behind it.",
    "LLM Engineering": "Your speciality is building with LLMs: prompting, streaming, tool calling, embeddings, RAG, fine-tuning and agent design.",
    "Web development": "Your speciality is web development: HTTP, REST APIs, JavaScript, React and browser behaviour.",
    "DevOps & Cloud": "Your speciality is DevOps and cloud: Linux, Docker, CI/CD, infrastructure as code, AWS and observability.",
}


LEVELS = {
    "Beginner": "Your reader is new to this topic. Avoid unexplained jargon, use everyday analogies, keep the answer under about 250 words and end with one concrete next step to try.",
    "Intermediate": "Your reader is a working developer who is new to this particular topic. Assume general programming knowledge, and mention the practical pitfalls.",
    "Expert": "Your reader is a senior engineer. Skip the basics, go straight to the mechanism, the edge cases, the performance characteristics and the trade-offs. Be dense rather than chatty.",
}


def build_system_prompt(expertise, level):
    """Glue the three pieces together into the system message for this conversation."""
    return "\n\n".join([
        BASE_SYSTEM_PROMPT,
        EXPERTISE.get(expertise, EXPERTISE["General technology"]),
        LEVELS.get(level, LEVELS["Intermediate"]),
    ])

In [5]:
# Have a look at what the model actually receives

print(build_system_prompt("Python", "Beginner"))

You are Tech Tutor, an expert technical assistant used during an LLM engineering course.

How you answer:
- Explain step by step: the idea first, then the detail.
- Use plain language. Introduce jargon only after you have defined it.
- Include a short, runnable code example whenever it makes the answer clearer.
- If the question is ambiguous, state the assumption you are making and answer anyway.
- If you do not know, say so instead of guessing.
- Always reply in markdown.

You have tools available. Use them when they make your answer more accurate:
- lookup_reference(topic): a curated cheat sheet plus a link to the official documentation.
  Use it when the user asks about a core language or library concept.
- run_python_snippet(code): actually executes a short, self-contained Python snippet and returns
  what it printed. Prefer to run a snippet before explaining what it does, then quote the real output.

Never invent tool output, and never claim you ran code that you did not run.

You

## 3. The tools (the bonus points)

**Tool 1 - `lookup_reference`** is the safe, day-4-style tool: a lookup table that returns a curated
summary and the official documentation link, so the assistant cites a real URL instead of hallucinating one.

**Tool 2 - `run_python_snippet`** is the fun one: the assistant can *run* a snippet and read the real
output before explaining it. That turns "here is what I think this code does" into "here is what this
code did".

> ⚠️ **A word of caution on tool 2.** It calls `exec` on code the model wrote, in this kernel. That is fine
> for a local learning notebook, but never expose it to untrusted users or point it at anything you care
> about. If you would rather not run model-written code at all, drop `run_snippet_tool` from the `tools`
> list a few cells down and everything else still works.

In [6]:
# Tool 1: a small curated reference table

REFERENCE = {
    "generator": (
        "A function that uses `yield` instead of `return`. Calling it returns a lazy iterator: the body runs "
        "up to the first `yield`, hands a value back, and resumes from there on the next `next()` call. Nothing "
        "is computed until it is asked for, so generators use constant memory over huge sequences.",
        "https://docs.python.org/3/glossary.html#term-generator",
    ),
    "yield from": (
        "`yield from iterable` delegates to another iterable: it yields every item that iterable produces, one at "
        "a time, without writing an explicit `for` loop. It also forwards `send()` and exceptions to a sub-generator.",
        "https://docs.python.org/3/reference/expressions.html#yield-expressions",
    ),
    "set comprehension": (
        "`{expr for item in iterable if condition}` builds a set in one expression. Because it is a set, duplicates "
        "are removed and ordering is not guaranteed - which is usually the reason to pick it over a list comprehension.",
        "https://docs.python.org/3/tutorial/datastructures.html#sets",
    ),
    "list comprehension": (
        "`[expr for item in iterable if condition]` builds a list in one expression. Faster and clearer than an "
        "append loop, but it materialises every element - use a generator expression `(...)` for large data.",
        "https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions",
    ),
    "decorator": (
        "A callable that takes a function and returns a replacement, applied with `@name` above a `def`. Used to add "
        "behaviour - caching, timing, retries, registration - without editing the function body.",
        "https://docs.python.org/3/glossary.html#term-decorator",
    ),
    "context manager": (
        "An object with `__enter__` / `__exit__`, used via `with`. It guarantees the cleanup step runs even if the "
        "block raises - closing files, releasing locks, committing or rolling back transactions.",
        "https://docs.python.org/3/reference/datamodel.html#context-managers",
    ),
    "async": (
        "`async def` declares a coroutine and `await` suspends it while waiting on I/O, letting the event loop run "
        "other work. It gives concurrency, not parallelism: CPU-bound work still blocks the loop.",
        "https://docs.python.org/3/library/asyncio.html",
    ),
    "gil": (
        "CPython's Global Interpreter Lock lets only one thread execute Python bytecode at a time. Threads still help "
        "with I/O-bound work (the lock is released while waiting); for CPU-bound work use multiprocessing.",
        "https://docs.python.org/3/glossary.html#term-global-interpreter-lock",
    ),
    "type hint": (
        "Optional annotations such as `def f(x: int) -> str:`. Ignored at runtime, but they drive editor autocomplete "
        "and static checkers like mypy, and they document intent.",
        "https://docs.python.org/3/library/typing.html",
    ),
    "openai streaming": (
        "Pass `stream=True` to `chat.completions.create` and you get an iterator of chunks instead of one response. "
        "Each chunk carries a delta; concatenate `chunk.choices[0].delta.content or ''` to build the answer as it arrives.",
        "https://platform.openai.com/docs/api-reference/chat/streaming",
    ),
    "openai tools": (
        "Describe a function with a JSON schema and pass it in `tools=`. If the model wants it, the response has "
        "`finish_reason == 'tool_calls'`; you run the function, append a message with `role='tool'` and the matching "
        "`tool_call_id`, then call the API again so the model can use the result.",
        "https://platform.openai.com/docs/guides/function-calling",
    ),
    "gradio": (
        "A Python library that turns a function into a web UI. `gr.ChatInterface` wraps a chat function; `gr.Blocks` "
        "gives full layout control. If the function is a generator, Gradio streams each yielded value to the browser.",
        "https://www.gradio.app/docs",
    ),
}


def lookup_reference(topic):
    """Look a technical topic up in the curated table above."""
    print(f"[TOOL] lookup_reference({topic!r})", flush=True)
    key = (topic or "").strip().lower()

    match = REFERENCE.get(key)
    if match is None:
        # be forgiving: "python generators" should still find "generator"
        for name, entry in REFERENCE.items():
            if name in key or key in name:
                match = entry
                break

    if match is None:
        return (
            f"No cheat sheet entry for '{topic}'. Answer from your own knowledge instead, and do not cite a "
            f"documentation URL unless you are certain of it. Topics that do have an entry: {', '.join(REFERENCE)}."
        )

    summary, url = match
    return f"Reference for '{topic}':\n{summary}\nOfficial documentation: {url}"

In [7]:
# Try it directly, before any LLM is involved

print(lookup_reference("yield from"))
print()
print(lookup_reference("python generators"))

[TOOL] lookup_reference('yield from')
Reference for 'yield from':
`yield from iterable` delegates to another iterable: it yields every item that iterable produces, one at a time, without writing an explicit `for` loop. It also forwards `send()` and exceptions to a sub-generator.
Official documentation: https://docs.python.org/3/reference/expressions.html#yield-expressions

[TOOL] lookup_reference('python generators')
Reference for 'python generators':
A function that uses `yield` instead of `return`. Calling it returns a lazy iterator: the body runs up to the first `yield`, hands a value back, and resumes from there on the next `next()` call. Nothing is computed until it is asked for, so generators use constant memory over huge sequences.
Official documentation: https://docs.python.org/3/glossary.html#term-generator


In [8]:
# Tool 2: actually run a snippet.
# NOTE: this exec's model-written code in this kernel - fine locally, never for untrusted input.

MAX_SNIPPET_CHARS = 2000
MAX_OUTPUT_CHARS = 2000


def run_python_snippet(code):
    """Execute a short, self-contained Python snippet and return whatever it printed."""
    print(f"[TOOL] run_python_snippet - {len(code or '')} chars", flush=True)

    if not code or not code.strip():
        return "No code was supplied, so nothing was run."
    if len(code) > MAX_SNIPPET_CHARS:
        return f"Refused to run: the snippet is longer than {MAX_SNIPPET_CHARS} characters. Send a smaller example."

    buffer = io.StringIO()
    namespace = {}
    try:
        with redirect_stdout(buffer):
            exec(code, namespace)
    except Exception as e:
        printed = buffer.getvalue().strip()
        result = f"The snippet raised {type(e).__name__}: {e}"
        if printed:
            result += f"\nIt printed this before failing:\n{printed}"
        return result

    printed = buffer.getvalue().strip()
    if not printed:
        return "The snippet ran without error but printed nothing - add a print() to see a value."
    if len(printed) > MAX_OUTPUT_CHARS:
        printed = printed[:MAX_OUTPUT_CHARS] + "\n... (output truncated)"
    return f"The snippet ran successfully and printed:\n{printed}"

In [9]:
# The week 1 question, checked by actually running it

print(run_python_snippet("""
books = [{"author": "Ada"}, {"author": "Grace"}, {"author": "Ada"}, {"title": "No author here"}]

def authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

print(sorted(authors(books)))
"""))

[TOOL] run_python_snippet - 226 chars
The snippet ran successfully and printed:
['Ada', 'Grace']


In [10]:
# Now describe both functions to the model with JSON schemas

lookup_tool = {
    "name": "lookup_reference",
    "description": (
        "Look up a curated cheat sheet and the official documentation URL for a core programming or "
        "LLM-engineering concept. Call this before explaining a well-known concept so that any link you cite is real."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The concept to look up, for example 'yield from', 'decorator' or 'openai streaming'",
            },
        },
        "required": ["topic"],
        "additionalProperties": False,
    },
}

run_snippet_tool = {
    "name": "run_python_snippet",
    "description": (
        "Execute a short, self-contained Python snippet and return exactly what it printed. Use this to check what "
        "a piece of code really does before you explain it. The snippet must print its result and must not need "
        "network access, user input or third-party packages."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "The Python source to execute. Include print() calls so the output is visible.",
            },
        },
        "required": ["code"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": lookup_tool},
    {"type": "function", "function": run_snippet_tool},
]

# name -> python function, so the handler below does not need an if/elif chain
TOOL_FUNCTIONS = {
    "lookup_reference": lookup_reference,
    "run_python_snippet": run_python_snippet,
}

In [11]:
def handle_tool_calls(tool_calls):
    """Run every tool the model asked for and build the matching 'tool' messages."""
    results = []
    for tool_call in tool_calls:
        name = tool_call["function"]["name"]
        try:
            arguments = json.loads(tool_call["function"]["arguments"] or "{}")
            content = TOOL_FUNCTIONS[name](**arguments)
        except Exception as e:
            # never let a bad tool call kill the conversation - report it back to the model instead
            content = f"The tool '{name}' failed with {type(e).__name__}: {e}"
        results.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call["id"],
        })
    return results

## 4. Streaming *and* tool calling in the same call

Day 4 used tools without streaming, and day 3 used streaming without tools. Doing both at once takes one extra step,
because when you stream, a tool call arrives in fragments too — the function name in one chunk, the JSON arguments a
few characters at a time across many more.

So the loop below does three things:

1. accumulates `delta.content` and yields it, so text appears as it is generated;
2. accumulates `delta.tool_calls` by index, re-assembling each call's `id`, `name` and `arguments`;
3. if any tool calls were assembled, runs them, appends the results, and loops round to stream the model's follow-up.

The `while True` matters: after seeing a tool result the model is allowed to call another tool, and this keeps going
until it finally replies with text.

In [12]:
def stream_answer(messages, model):
    """Stream a reply for `messages`, running any tools the model asks for along the way.

    Yields the answer so far, so the caller can just keep overwriting what it displays.
    """
    answer_so_far = ""

    while True:
        stream = openai.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            stream=True,
        )

        content = ""
        partial_calls = {}   # index -> the tool call being assembled

        for chunk in stream:
            if not chunk.choices:
                continue                      # usage-only chunks have no choices
            delta = chunk.choices[0].delta

            if delta.content:
                content += delta.content
                yield answer_so_far + content

            for tc in (delta.tool_calls or []):
                call = partial_calls.setdefault(
                    tc.index,
                    {"id": "", "type": "function", "function": {"name": "", "arguments": ""}},
                )
                if tc.id:
                    call["id"] = tc.id
                if tc.function and tc.function.name:
                    call["function"]["name"] += tc.function.name
                if tc.function and tc.function.arguments:
                    call["function"]["arguments"] += tc.function.arguments

        if not partial_calls:
            return                            # plain text answer - we are done

        tool_calls = [partial_calls[i] for i in sorted(partial_calls)]
        names = ", ".join(call["function"]["name"] for call in tool_calls)

        answer_so_far += content
        if answer_so_far and not answer_so_far.endswith("\n\n"):
            answer_so_far += "\n\n"
        answer_so_far += f"*\U0001F527 called `{names}`*\n\n"
        yield answer_so_far                   # show the user that a tool is running

        messages = messages + [{"role": "assistant", "content": content or None, "tool_calls": tool_calls}]
        messages = messages + handle_tool_calls(tool_calls)

In [13]:
def chat(message, history, model_label=DEFAULT_MODEL, expertise="General technology", level="Intermediate"):
    """The function Gradio calls. Same shape as day 3, plus the three dropdown values."""
    system_prompt = build_system_prompt(expertise, level)
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    yield from stream_answer(messages, MODELS[model_label])

In [15]:
# Quick check in the notebook before wiring up any UI - streaming straight into the output cell

from IPython.display import Markdown, display, update_display

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

handle = display(Markdown(""), display_id=True)
for partial in chat(question, [], DEFAULT_MODEL, "Python", "Beginner"):
    update_display(Markdown(partial), display_id=handle.display_id)

This code snippet uses a Python feature called "generator expression" along with the `yield from` statement. Let's break it down step by step.

### Step 1: Understanding the Components

1. **`{book.get("author") for book in books if book.get("author")}`**: This part creates a set comprehension. 
   - It loops over the collection `books`.
   - For each `book`, it tries to get the value associated with the key `"author"` using `book.get("author")`.
   - It includes the author in the set only if the author exists (i.e., it's not `None`).
   - The output is a set of unique authors from the `books`.

2. **`yield from`**: This is a way to yield values from a generator. It allows the current generator to yield all values from another generator or an iterable. In this case, it yields all the authors collected from the set comprehension.

### Step 2: What This Code Does

Combining both parts, this code snippet effectively:
- Collects all unique author names from the list of `books`, ignoring any entries where the author is not present.
- Yields each author's name one by one.

### Short Example

Here’s an example with a sample list of books:

```python
books = [
    {'title': 'Book 1', 'author': 'Author A'},
    {'title': 'Book 2', 'author': 'Author B'},
    {'title': 'Book 3', 'author': None},
    {'title': 'Book 4', 'author': 'Author A'},
]

def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Example usage
for author in get_authors(books):
    print(author)
```

### Next Step

Try running the example code to see how it collects and yields unique authors from the sample `books`. You can modify the `books` list to add more entries and see how it handles different cases!

## 5. The Gradio UI

### 5a. The quick version

`gr.ChatInterface` gives a working chat app from the `chat` function above in one line. The `additional_inputs` are
the three dropdowns; Gradio passes them as the extra arguments and tucks them into a collapsible panel.

Because `chat` is a generator, Gradio streams every yielded value into the chat bubble for free.

In [16]:
view = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Tech Tutor",
    description="Ask a technical question. Pick the model, the speciality and who the answer is for.",
    additional_inputs=[
        gr.Dropdown(list(MODELS), label="Model", value=DEFAULT_MODEL),
        gr.Dropdown(list(EXPERTISE), label="Speciality", value="General technology"),
        gr.Radio(list(LEVELS), label="Explain for", value="Intermediate"),
    ],
    examples=[
        ["What does `yield from {b.get('author') for b in books if b.get('author')}` do?"],
        ["Why is my pandas groupby so slow on 10 million rows?"],
        ["What is the difference between a process and a thread in Python?"],
    ],
)

view.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### 5b. The full prototype

Same engine, but laid out with `gr.Blocks` so the controls are always visible next to the conversation.

The event wiring is the usual two-step: `do_entry` appends the user's message and clears the textbox immediately,
then `.then(...)` runs `stream_bot`, which rewrites the last assistant bubble on every yield.

In [17]:
def do_entry(message, history):
    """Step 1 - show the user's message straight away and clear the box."""
    if not message or not message.strip():
        return "", history
    return "", history + [{"role": "user", "content": message}]


def stream_bot(history, model_label, expertise, level):
    """Step 2 - stream the reply into a new assistant bubble."""
    if not history or history[-1]["role"] != "user":
        yield history
        return

    message = history[-1]["content"]
    prior = history[:-1]                       # everything before this question
    history = history + [{"role": "assistant", "content": ""}]

    for partial in chat(message, prior, model_label, expertise, level):
        history[-1] = {"role": "assistant", "content": partial}
        yield history

In [18]:
with gr.Blocks(title="Tech Tutor", theme=gr.themes.Soft()) as ui:
    gr.Markdown("# \U0001F6E0\uFE0F Tech Tutor\nYour technical question answerer - streaming, switchable models, and real tools.")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=460, type="messages", label="Conversation")
            with gr.Row():
                entry = gr.Textbox(
                    label="Your question",
                    placeholder="Ask anything technical, or paste some code and ask what it does...",
                    lines=3,
                    scale=5,
                )
                send = gr.Button("Ask", variant="primary", scale=1)
            clear = gr.Button("Clear conversation")

        with gr.Column(scale=1):
            model_dropdown = gr.Dropdown(list(MODELS), label="Model", value=DEFAULT_MODEL)
            expertise_dropdown = gr.Dropdown(list(EXPERTISE), label="Speciality", value="Python")
            level_radio = gr.Radio(list(LEVELS), label="Explain for", value="Intermediate")
            gr.Markdown(
                "**Tools available**\n\n"
                "- `lookup_reference` - curated cheat sheet + official docs link\n"
                "- `run_python_snippet` - actually runs the code, then explains the real output\n\n"
                "A *\U0001F527 called ...* line appears in the answer whenever a tool fires."
            )

    gr.Examples(
        examples=[
            'Please explain what this code does and why:\nyield from {book.get("author") for book in books if book.get("author")}',
            "What does this print, and why?\nprint([x for x in range(5)][::-1])",
            "Explain decorators, and show me one that times a function.",
            "How does streaming work with the OpenAI chat completions API?",
        ],
        inputs=entry,
        label="Try one of these",
    )

    stream_inputs = [chatbot, model_dropdown, expertise_dropdown, level_radio]
    entry.submit(do_entry, [entry, chatbot], [entry, chatbot], queue=False).then(stream_bot, stream_inputs, chatbot)
    send.click(do_entry, [entry, chatbot], [entry, chatbot], queue=False).then(stream_bot, stream_inputs, chatbot)
    clear.click(lambda: [], None, chatbot, queue=False)

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[TOOL] run_python_snippet - 34 chars


## What to try

- Paste the week 1 question and watch `run_python_snippet` fire - the answer quotes output the model genuinely observed.
- Ask the same question at **Beginner** and then at **Expert**: only the system prompt changed.
- Ask *"what does `yield from` do?"* and watch `lookup_reference` supply the real documentation link.
- Switch to **GPT-4.1-nano** for the speed difference, and **GPT-4o** when a question is genuinely hard.

**Possible next steps:** add audio in and out with `openai.audio.transcriptions` / `openai.audio.speech`, keep a
conversation log so the tutor remembers across sessions, or add a tool that searches the course notebooks.